# Immunotherapy Patient Extraction
Filters medication orders for a target list of immunotherapy drugs and writes two output sheets:
- **Orders** – one row per order
- **Summary** – one row per patient × drug, with earliest start and latest end date

In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
INPUT_FILE       = "your_file.csv"        # change to your input file name
OUTPUT_FILE      = None                   # None → auto-named  <input>_immunotherapy.xlsx
DEMOGRAPHIC_FILE = "demographic.csv"      # CSV with IP_PATIENT_ID, AGE, SEX
ID_LOOKUP_FILE   = "ID_LOOKUP.csv"        # CSV with IP_PATIENT_ID, Anon Patient Name, Anon MRN

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_csv(INPUT_FILE)
df.columns = df.columns.str.strip()
print(f"Loaded {len(df):,} rows — columns: {list(df.columns)}")

In [ ]:
# ── Filter immunotherapy orders ───────────────────────────────────────────────
# Match all *mab drugs, exclude denosumab
mask = (
    df["EPIC_MEDICATION_NAME"].str.contains(r"mab\b", case=False, na=False) &
    ~df["EPIC_MEDICATION_NAME"].str.contains("denosumab", case=False, na=False)
)
filtered = df[mask].copy()

print(f"Filtered rows   : {len(filtered):,}")
print(f"Unique patients : {filtered['IP_PATIENT_ID'].nunique():,}")
filtered.head()

In [ ]:
# ── Sheet 1 – Orders (one row per order) ─────────────────────────────────────
sheet1_cols = ["IP_PATIENT_ID", "ORDER_DATE", "START_DATE", "END_DATE",
               "EPIC_MED_ID", "EPIC_MEDICATION_NAME"]
sheet1_cols = [c for c in sheet1_cols if c in filtered.columns]

sheet1 = filtered[sheet1_cols].reset_index(drop=True)
sheet1.head()

In [ ]:
# ── Sheet 2 – Summary (split into episodes if gap > 1 month) ─────────────────
import warnings

work = filtered.copy()
work["START_DATE"] = pd.to_datetime(work["START_DATE"])
work["END_DATE"]   = pd.to_datetime(work["END_DATE"])
work = work.sort_values(["IP_PATIENT_ID", "EPIC_MEDICATION_NAME", "START_DATE"])

def assign_episodes(grp):
    grp = grp.sort_values("START_DATE").copy()
    episode = 0
    prev_end = None
    episodes = []
    for _, row in grp.iterrows():
        if prev_end is None or (row["START_DATE"] - prev_end).days > 31:
            episode += 1
        episodes.append(episode)
        prev_end = row["END_DATE"] if prev_end is None else max(prev_end, row["END_DATE"])
    grp["EPISODE"] = episodes
    return grp

group_cols = [c for c in ["IP_PATIENT_ID", "EPIC_MEDICATION_NAME"] if c in work.columns]
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    work = work.groupby(group_cols, group_keys=False).apply(assign_episodes)

work.head()

In [ ]:
# ── Aggregate episodes into sheet2 ───────────────────────────────────────────
agg_cols = [c for c in ["IP_PATIENT_ID", "EPIC_MED_ID", "EPIC_MEDICATION_NAME", "EPISODE"]
            if c in work.columns]
sheet2 = (
    work.groupby(agg_cols, as_index=False)
    .agg(START_DATE=("START_DATE", "min"), END_DATE=("END_DATE", "max"))
    .sort_values(["IP_PATIENT_ID", "EPIC_MEDICATION_NAME", "EPISODE"])
    .drop(columns=["EPISODE"])
    .reset_index(drop=True)
)
sheet2.head()

In [ ]:
# ── Merge demographics and anonymized IDs into sheet2 ────────────────────────
demo = pd.read_csv(DEMOGRAPHIC_FILE, usecols=["IP_PATIENT_ID", "AGE", "SEX"])
sheet2 = sheet2.merge(demo, on="IP_PATIENT_ID", how="left")

id_lookup = pd.read_csv(ID_LOOKUP_FILE, usecols=["IP_PATIENT_ID", "Anon Patient Name", "Anon MRN"])
id_lookup = id_lookup.drop_duplicates("IP_PATIENT_ID")
# Extract numeric part from "PATIENT^380977672" → "380977672"
id_lookup["Anon Patient Name"] = id_lookup["Anon Patient Name"].str.split("^").str[-1]
sheet2 = sheet2.merge(id_lookup, on="IP_PATIENT_ID", how="left")

# Move new columns to appear right after IP_PATIENT_ID
cols = sheet2.columns.tolist()
for col in ["Anon MRN", "Anon Patient Name", "SEX", "AGE"]:
    if col in cols:
        cols.insert(1, cols.pop(cols.index(col)))
sheet2 = sheet2[cols]
sheet2.head()

In [ ]:
# ── Write output ──────────────────────────────────────────────────────────────
stem = Path(INPUT_FILE).stem if OUTPUT_FILE is None else Path(OUTPUT_FILE).stem

xlsx_file    = stem + "_immunotherapy.xlsx"
csv_orders   = stem + "_immunotherapy_orders.csv"
csv_summary  = stem + "_immunotherapy_summary.csv"

with pd.ExcelWriter(xlsx_file, engine="openpyxl", datetime_format="YYYY-MM-DD") as writer:
    sheet1.to_excel(writer, sheet_name="Orders",  index=False)
    sheet2.to_excel(writer, sheet_name="Summary", index=False)

sheet1.to_csv(csv_orders,  index=False)
sheet2.to_csv(csv_summary, index=False)

print(f"Excel  : {xlsx_file}")
print(f"CSV    : {csv_orders}")
print(f"CSV    : {csv_summary}")

In [ ]:
# ── Save deduplicated ID lookup ───────────────────────────────────────────────
id_lookup_dedup = (
    pd.read_csv(ID_LOOKUP_FILE, usecols=["IP_PATIENT_ID", "Anon Patient Name", "Anon MRN"])
    .drop_duplicates("IP_PATIENT_ID")
    .reset_index(drop=True)
)
id_lookup_dedup["Anon Patient Name"] = id_lookup_dedup["Anon Patient Name"].str.split("^").str[-1]
out_path = Path(ID_LOOKUP_FILE).stem + "_unique.csv"
id_lookup_dedup.to_csv(out_path, index=False)
print(f"Saved {len(id_lookup_dedup):,} unique patients to {out_path}")

In [ ]:
# ── IDs in lookup but not in sheet2 ──────────────────────────────────────────
lookup_df = pd.read_csv(ID_LOOKUP_FILE, usecols=["IP_PATIENT_ID", "Anon MRN"]).drop_duplicates("IP_PATIENT_ID")
sheet2_ids = set(sheet2["IP_PATIENT_ID"].unique())
missing = lookup_df[~lookup_df["IP_PATIENT_ID"].isin(sheet2_ids)].reset_index(drop=True)
print(f"{len(missing)} IDs in lookup not found in sheet2:")
print(missing.to_string(index=False))